# 🎬 Movie Recommendation System

## Project Overview
Built a content-based movie recommender system that suggests 
similar movies based on genre, keywords, cast, and director.

## How It Works
1. Combined all movie features into one text column
2. Converted text to numbers using TF-IDF Vectorizer
3. Measured similarity between all 4760 movies using Cosine Similarity
4. Returns top 5 most similar movies for any input

## Example Results
- The Dark Knight → Batman Begins, Dark Knight Rises...
- The Avengers → Age of Ultron, Iron Man 2...
- Annabelle → Seed of Chucky, Child's Play 2...

## Tools Used
Python, Pandas, Scikit-learn, TF-IDF, Cosine Similarity

## Author
**Faith Mwende** | [GitHub](https://github.com/Mwende-Fifi)

In [1]:
# import libraries
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# Loading the dataset
url = "https://raw.githubusercontent.com/YBI-Foundation/Dataset/main/Movies%20Recommendation.csv"
df = pd.read_csv(url)

print("Shape:", df.shape)
print("\nColumns:", df.columns.tolist())

Shape: (4760, 21)

Columns: ['Movie_ID', 'Movie_Title', 'Movie_Genre', 'Movie_Language', 'Movie_Budget', 'Movie_Popularity', 'Movie_Release_Date', 'Movie_Revenue', 'Movie_Runtime', 'Movie_Vote', 'Movie_Vote_Count', 'Movie_Homepage', 'Movie_Keywords', 'Movie_Overview', 'Movie_Production_House', 'Movie_Production_Country', 'Movie_Spoken_Language', 'Movie_Tagline', 'Movie_Cast', 'Movie_Crew', 'Movie_Director']


In [4]:
# Selecting relevant columns and clean
df = df[['Movie_Title', 'Movie_Genre', 'Movie_Keywords', 
         'Movie_Overview', 'Movie_Cast', 'Movie_Director']]

# Fill missing values with empty string
df = df.fillna('')
print("Shape:", df.shape)
print("\nSample movie:")
print(df.iloc[0])

Shape: (4760, 6)

Sample movie:
Movie_Title                                              Four Rooms
Movie_Genre                                            Crime Comedy
Movie_Keywords            hotel new year's eve witch bet hotel room
Movie_Overview    It's Ted the Bellhop's first night on the job....
Movie_Cast        Tim Roth Antonio Banderas Jennifer Beals Madon...
Movie_Director                                       Allison Anders
Name: 0, dtype: str


In [5]:
# Combine all features into one text column
df['Combined_Features'] = (df['Movie_Genre'] + ' ' +
                              df['Movie_Keywords'] + ' ' +
                              df['Movie_Overview'] + ' ' + 
                              df['Movie_Cast'] + ' ' +
                              df['Movie_Director'])
print("Sample combined features:")
print(df['Combined_Features'][0])

Sample combined features:
Crime Comedy hotel new year's eve witch bet hotel room It's Ted the Bellhop's first night on the job...and the hotel's very unusual guests are about to place him in some outrageous predicaments. It seems that this evening's room service is serving up one unbelievable happening after another. Tim Roth Antonio Banderas Jennifer Beals Madonna Marisa Tomei Allison Anders


In [6]:
# Convert text to numbers using TF-IDF
tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['Combined_Features'])

print("TF-IDF Matrix Shape:", tfidf_matrix.shape)

TF-IDF Matrix Shape: (4760, 29644)


In [7]:
# Calculating similarity between all movies
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
print("Similarity Matrix Shape:", cosine_sim.shape)

Similarity Matrix Shape: (4760, 4760)


In [8]:
# Bulding the recommender function
# Create a mapping of movie titles to index
indices = pd.Series(df.index, index=df['Movie_Title']).drop_duplicates()

def get_recommendations(movie_title, num_recommendations=5):
    # Get index of the movie
    idx = indices[movie_title]

    # Get similarity scores for this movie
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort by similarity score
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    # Get top 5 most similar (excluding itself)
    sim_scores = sim_scores[1:num_recommendations+1]

    # Get movie indices
    movie_indices = [i[0] for i in sim_scores]

    return df['Movie_Title'].iloc[movie_indices].tolist()

In [9]:
# Tesing the recommender
movie = "The Dark Knight"
recommendations = get_recommendations(movie)

print(f"Because you watched '{movie}', we recommend:")
for i, rec in enumerate(recommendations, 1):
    print(f"{i}. {rec}")

Because you watched 'The Dark Knight', we recommend:
1. The Dark Knight Rises
2. Batman Begins
3. Batman Returns
4. Batman Forever
5. Batman: The Dark Knight Returns, Part 2


In [10]:
# Test with another movie
movie = "The Avengers"
recommendations = get_recommendations(movie)

print(f"Because you watched '{movie}', we recommend:")
for i, rec in enumerate(recommendations, 1):
    print(f"{i}. {rec}")

Because you watched 'The Avengers', we recommend:
1. Avengers: Age of Ultron
2. Iron Man 2
3. Captain America: The Winter Soldier
4. Captain America: Civil War
5. X-Men


In [14]:
# Interactive Recommender
movie = input("Enter a movie name: ")

if movie in indices:
    recommendations = get_recommendations(movie)
    print(f"\nBecause you watched '{movie}', we recommend:")
    for i, rec in enumerate(recommendations, 1):
        print(f"{i}. {rec}")
else:
    print(f"Sorry, '{movie}' not found in our database!")
    print("\nTry one of these:")
    print(df['Movie_Title'].sample(5).tolist())


Because you watched 'Annabelle', we recommend:
1. Seed of Chucky
2. Bride of Chucky
3. Loving Annabelle
4. Child's Play 2
5. The Nutcracker: The Untold Story
